# Bootstrap Your Own Latent (BYOL)
## A New Approach to Self-Supervised Learning

**Paper:** Grill, Strub, Altché, Tallec, Richemond, Buchatskaya, Doersch, Pires, Guo, Azar, Piot, Kavukcuoglu, Munos, Valko  
**Conference:** NeurIPS 2020  
**arXiv:** [2006.07733](https://arxiv.org/abs/2006.07733)  
**Official Code:** [google-deepmind/deepmind-research/byol](https://github.com/google-deepmind/deepmind-research/tree/master/byol) (JAX/Haiku)

---

### Research Goal
Reproduce the BYOL self-supervised pretraining framework from scratch in PyTorch. Train on CIFAR-10 with a ResNet-18 backbone. Evaluate representation quality via linear probing and k-NN classification.

### Central Research Question
> **Can self-supervised learning produce useful visual representations without negative pairs, and if so, why doesn't the model collapse to a trivial solution?**

---

### Table of Contents
0. [Setup & Environment](#0)
1. [Historical Context & Motivation](#1)
2. [Paper Walkthrough](#2)
3. [Mathematical Framework](#3)
4. [Implementation](#4)
5. [Training](#5)
6. [Evaluation](#6)
7. [Visualizations](#7)
8. [Ablation Studies](#8)
9. [Critical Review](#9)
10. [Reflection & Future Work](#10)

<a id='0'></a>
## 0. Setup & Environment

This notebook is designed to run on **Google Colab Free Tier** (T4 GPU).  
Total runtime: **~20–25 minutes** (50 epochs main training + lightweight ablations).

**Persistence:** All data, checkpoints, and results are saved to Google Drive so you can safely close the tab, get disconnected, or run out of session time — training will automatically resume from the last checkpoint.

In [ ]:
# ─── Mount Google Drive for Persistent Storage ───
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('✓ Google Drive mounted')
except ImportError:
    IN_COLAB = False
    print('⚠ Not running in Colab — using local storage')

# ─── Persistent Directories ───
if IN_COLAB:
    DRIVE_BASE = '/content/drive/MyDrive/BYOL'
else:
    DRIVE_BASE = './BYOL_output'

DATA_DIR = os.path.join(DRIVE_BASE, 'data')
CHECKPOINT_DIR = os.path.join(DRIVE_BASE, 'checkpoints')
RESULTS_DIR = os.path.join(DRIVE_BASE, 'results')

for d in [DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Data directory:       {DATA_DIR}')
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Results directory:    {RESULTS_DIR}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from tqdm.auto import tqdm
import copy
import math
import time
import glob
import json
import warnings
warnings.filterwarnings('ignore')

# ─── Reproducibility ───
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = False  # True hurts perf; not needed for SSL
torch.backends.cudnn.benchmark = True        # Auto-tune convolution kernels

# ─── Device ───
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'  # AMP only works on CUDA
print(f'Device: {device}')
print(f'AMP enabled: {USE_AMP}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

<a id='1'></a>
## 1. Historical Context & Motivation

### The Batch Size Problem in Contrastive Learning

In Week 1, I implemented **SimCLR** and identified its core limitation: the NT-Xent loss uses $2N - 2$ in-batch negatives, so representation quality scales directly with batch size. SimCLR needs $N \geq 4096$ to work well — requiring multi-TPU infrastructure that most researchers will never touch.

**MoCo** (He et al., 2020) partially solved this by maintaining a queue of 65,536 past representations as negatives, decoupling the number of negatives from the batch size. But MoCo still uses a contrastive loss — it still needs negatives.

### BYOL's Radical Departure

BYOL asks a question that, at the time, most researchers would have dismissed:

> *What if we don't use negatives at all?*

The immediate objection is **representational collapse**: without a repulsive force pushing different images apart, the model could trivially minimize any positive-pair loss by mapping every input to the same constant vector. Loss = 0, representations = useless.

BYOL's contribution is showing that collapse can be prevented through **architectural asymmetry** rather than explicit negative examples:
- An **online network** with a predictor head
- A **target network** without a predictor, updated via exponential moving average
- A **stop-gradient** on the target

The result: 74.3% top-1 on ImageNet linear eval (ResNet-50), matching or exceeding contrastive methods, with no negatives and no batch size dependence.

> 🔬 **Researcher's Note:** The intellectual leap here is subtle. MoCo and SimCLR were both searching for *better ways to supply negatives*. BYOL reframes the problem entirely — negatives were never the signal, they were a regularizer. The actual learning signal comes from predicting one view from another.

<a id='2'></a>
## 2. Paper Walkthrough

### Architecture

```
                    Image x
                   /       \
               aug_1       aug_2
                 |           |
                 v           v
             view v       view v'
                 |           |
         ┌───────┴───┐  ┌───┴───────┐
         │  ONLINE    │  │  TARGET    │
         │  NETWORK   │  │  NETWORK   │
         │            │  │            │
         │  Encoder   │  │  Encoder   │
         │  f_θ       │  │  f_ξ       │
         │    ↓       │  │    ↓       │
         │  Projector │  │  Projector │
         │  g_θ       │  │  g_ξ       │
         │    ↓       │  │    ↓       │
         │  Predictor │  │  (none)    │
         │  q_θ       │  │            │
         │    ↓       │  │    ↓       │
         │   p_θ      │  │   z'_ξ     │
         └────┬───────┘  └────┬───────┘
              │               │
              └──── Loss ─────┘
                 (MSE on L2-normed)

    Target parameters: ξ ← τξ + (1-τ)θ  (EMA, no gradients)
```

### Key Design Decisions

| Component | Design Choice | Why |
|---|---|---|
| Predictor | Only on online network | Breaks symmetry → collapse is not a stable equilibrium |
| Target update | EMA (no gradients) | Provides slowly-evolving regression target |
| Loss | MSE on L2-normalized vectors | Equivalent to negative cosine similarity; prevents magnitude collapse |
| Symmetrization | Swap views, sum losses | Both views contribute equally; doubles the signal |
| Augmentation | Asymmetric (view1 ≠ view2) | Views are complementary, not just stochastic copies |

> ⚠️ **Common Mistake:** Forgetting the stop-gradient on the target network. If gradients flow through both networks, the system has a trivial solution where both collapse together. PyTorch handles this naturally since the target parameters aren't in the optimizer's parameter groups, but you must also detach the target outputs in the loss computation.

### The Three Collapse-Prevention Mechanisms

1. **Stop-gradient on target:** The target can't adapt to make the online network's job easier. The online network must learn genuinely useful features to predict the target's output.

2. **Predictor asymmetry:** The online network has more capacity (the predictor MLP) than the target. If both had identical architectures, the easiest solution would be for both to output the same constant — but the predictor adds a transformation that the target can't mirror.

3. **EMA momentum:** The target is a smoothed, historical version of the online network. It's always "behind" — when the online network changes, the target hasn't caught up yet. This creates a non-stationary regression target that the online network must continuously adapt to.

> 📓 **Research Diary:** I spent time trying to figure out which of these three is "the real" collapse-prevention mechanism. The answer is that it's the interaction of all three. The ablation studies (Section 5 of the paper) show that removing any one of them degrades performance significantly. This is a system, not a single trick.

<a id='3'></a>
## 3. Mathematical Framework

### The BYOL Loss

Given an image $x$, produce two augmented views $v = t(x)$ and $v' = t'(x)$.

**Online network** processes $v$:
$$y_\theta = f_\theta(v), \quad z_\theta = g_\theta(y_\theta), \quad q_\theta = q_\theta(z_\theta)$$

**Target network** processes $v'$:
$$y'_\xi = f_\xi(v'), \quad z'_\xi = g_\xi(y'_\xi)$$

L2-normalize the prediction and target projection:
$$\bar{q}_\theta = \frac{q_\theta}{\|q_\theta\|_2}, \quad \bar{z}'_\xi = \frac{z'_\xi}{\|z'_\xi\|_2}$$

The loss for one direction:
$$\mathcal{L}_{\theta,\xi} = \| \bar{q}_\theta - \bar{z}'_\xi \|_2^2$$

Expanding this on the unit sphere:
$$\| \bar{q} - \bar{z}' \|_2^2 = \|\bar{q}\|^2 + \|\bar{z}'\|^2 - 2\langle \bar{q}, \bar{z}' \rangle = 2 - 2 \cos(\bar{q}, \bar{z}')$$

So minimizing the L2 distance on L2-normalized vectors is equivalent to **maximizing cosine similarity**. The constant 2 doesn't affect gradients.

**Symmetrized loss** (swap views and sum):
$$\tilde{\mathcal{L}}_{\theta,\xi} = \mathcal{L}_{\theta,\xi} + \mathcal{L}'_{\theta,\xi}$$

where $\mathcal{L}'$ feeds $v'$ to online, $v$ to target.

### EMA Schedule

The decay rate $\tau$ follows a cosine schedule:
$$\tau_k = 1 - (1 - \tau_{\text{base}}) \cdot \left(\frac{\cos(\pi k / K) + 1}{2}\right)$$

where $k$ is the current step, $K$ is the total number of steps, and $\tau_{\text{base}} = 0.996$.

- At $k=0$: $\tau_0 = 1 - (1 - 0.996) \cdot 1 = 0.996$ (target updates relatively fast)
- At $k=K$: $\tau_K = 1 - (1 - 0.996) \cdot 0 = 1.0$ (target is frozen)

**Intuition:** Early in training, the online network is learning rapidly and the target needs to track it. Late in training, the online network has converged and the target should provide a very stable regression target.

### Why L2 Normalization is Essential

Without L2 normalization, the model can minimize $\| q - z' \|^2$ by simply shrinking all representations toward zero:
$$q \to \mathbf{0}, \quad z' \to \mathbf{0} \implies \text{loss} \to 0$$

L2 normalization projects everything onto the unit sphere, removing magnitude as a degree of freedom. The only way to reduce the loss is to **align the directions** of $q$ and $z'$, which requires learning meaningful features.

> ⚙️ **Engineering Decision:** In the implementation, I use `F.normalize(x, dim=-1)` which is equivalent to $x / \|x\|_2$ and handles the gradient correctly. Some implementations use `cosine_similarity` directly — both are mathematically equivalent, but MSE on normalized vectors makes the connection to the paper's notation more explicit.

### 4.0 Hyperparameters (Colab Free-Tier Optimized)

| Parameter | Value | Source |
|---|---|---|
| Backbone | ResNet-18 | Adapted (paper uses ResNet-50) |
| Projector hidden dim | 1024 | Reduced from paper's 4096 for T4 VRAM |
| Projector output dim | 256 | Paper |
| Predictor hidden dim | 1024 | Reduced from paper's 4096 for T4 VRAM |
| Predictor output dim | 256 | Paper |
| EMA base τ | 0.996 | Paper |
| Batch size | 128 | Adapted for T4 (paper uses 4096) |
| Epochs | 50 | Adapted (paper uses 1000) |
| Optimizer | AdamW | Adapted (paper uses LARS) |
| Learning rate | 1e-3 | Adapted for AdamW |
| Weight decay | 1.5e-6 | Paper |
| Warmup epochs | 5 | Adapted for fewer total epochs |
| Image size | 32×32 | CIFAR-10 native |

> 📊 **Colab Free-Tier Optimizations:** We reduce projector/predictor hidden dims from 4096→1024 (still effective for CIFAR-10 at 32×32 with ResNet-18's 512-dim features). Batch size is reduced from 256→128 to stay well within T4's 15GB VRAM. Epochs are reduced from 100→50 — on CIFAR-10, most representation learning happens in the first 40–50 epochs. These changes cut VRAM by ~60% and runtime to ~20–25 min on T4.

> 📊 **Paper vs. Ours:** The paper uses LARS optimizer with base LR=0.2 scaled linearly with batch size. At our smaller batch size, AdamW works well and is simpler to configure.

In [ ]:
# ─── Hyperparameters (Colab Free-Tier Optimized) ───

# Architecture — reduced from 4096 to 1024 (plenty for CIFAR-10 + ResNet-18)
PROJ_HIDDEN_DIM = 1024
PROJ_OUTPUT_DIM = 256
PRED_HIDDEN_DIM = 1024
PRED_OUTPUT_DIM = 256

# Training — smaller batch + fewer epochs for T4 GPU
BATCH_SIZE = 128
NUM_EPOCHS = 50
BASE_LR = 1e-3
WEIGHT_DECAY = 1.5e-6
WARMUP_EPOCHS = 5

# BYOL-specific
EMA_TAU_BASE = 0.996

# Evaluation
NUM_CLASSES = 10
KNN_K = 200

# Data
IMAGE_SIZE = 32
NUM_WORKERS = 2

# Checkpointing — more frequent saves for Colab stability
CHECKPOINT_EVERY = 5  # Save every 5 epochs for safe resume

print(f'Config: ResNet-18, BS={BATCH_SIZE}, Epochs={NUM_EPOCHS}, LR={BASE_LR}, τ_base={EMA_TAU_BASE}')
print(f'Checkpoints saved every {CHECKPOINT_EVERY} epochs to: {CHECKPOINT_DIR}')

# Report VRAM status
if device.type == 'cuda':
    free_mem = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f'Available VRAM: {free_mem / 1e9:.1f} GB')

### 4.1 Data Augmentation Pipeline

BYOL uses an **asymmetric** augmentation pipeline. View 1 and View 2 apply different probabilities for Gaussian blur and solarization. This is a deliberate design choice — it forces the two views to carry complementary information.

| Augmentation | View 1 | View 2 |
|---|---|---|
| Random Resized Crop | scale=(0.2, 1.0) | scale=(0.2, 1.0) |
| Horizontal Flip | p=0.5 | p=0.5 |
| Color Jitter | p=0.8 | p=0.8 |
| Grayscale | p=0.2 | p=0.2 |
| Gaussian Blur | **p=1.0** | **p=0.1** |
| Solarize | **p=0.0** | **p=0.2** |

> ⚙️ **Engineering Decision:** For CIFAR-10 (32×32), Gaussian blur with a large kernel can destroy too much information. I use a kernel size of 3 (vs. 23 for 224×224 ImageNet). The blur kernel size roughly scales as `image_size / 10`, following the official implementation.

In [ ]:
# ─── Augmentation Pipeline ───

class GaussianBlur:
    """Gaussian blur augmentation as used in BYOL.

    The kernel size is adapted for the input resolution.
    For CIFAR-10 (32x32), we use kernel_size=3.
    For ImageNet (224x224), the paper uses kernel_size=23.
    """
    def __init__(self, kernel_size=3, sigma_min=0.1, sigma_max=2.0):
        self.kernel_size = kernel_size
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max

    def __call__(self, img):
        sigma = np.random.uniform(self.sigma_min, self.sigma_max)
        return T.functional.gaussian_blur(img, self.kernel_size, sigma)


class Solarize:
    """Solarize augmentation: invert pixels above a threshold.

    Applied only to view 2 with p=0.2 in the paper.
    For PIL images, threshold is in [0, 256]. For tensors, in [0, 1].
    """
    def __init__(self, threshold=128):
        self.threshold = threshold

    def __call__(self, img):
        return T.functional.solarize(img, self.threshold)


def get_byol_augmentation(view='view1', image_size=32):
    """Build the augmentation pipeline for a given view.

    View 1: Always applies Gaussian blur, never solarizes.
    View 2: Rarely applies Gaussian blur, sometimes solarizes.

    This asymmetry is a deliberate design choice in BYOL.
    """
    # Both views share these base augmentations
    base_transforms = [
        T.RandomResizedCrop(image_size, scale=(0.2, 1.0)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomApply([
            T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)
        ], p=0.8),
        T.RandomGrayscale(p=0.2),
    ]

    # Blur kernel size adapted for resolution: image_size / 10, forced odd
    blur_kernel = max(3, (image_size // 10) | 1)

    if view == 'view1':
        # View 1: blur always, no solarize
        view_transforms = [
            T.RandomApply([GaussianBlur(kernel_size=blur_kernel)], p=1.0),
        ]
    elif view == 'view2':
        # View 2: blur rarely, solarize sometimes
        view_transforms = [
            T.RandomApply([GaussianBlur(kernel_size=blur_kernel)], p=0.1),
            T.RandomApply([Solarize(threshold=128)], p=0.2),
        ]
    else:
        raise ValueError(f'Unknown view: {view}')

    return T.Compose(base_transforms + view_transforms + [
        T.ToTensor(),
        T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                    std=[0.2470, 0.2435, 0.2616]),
    ])


class BYOLPairTransform:
    """Generates two augmented views of each image using BYOL's
    asymmetric augmentation pipeline.
    """
    def __init__(self, image_size=32):
        self.transform1 = get_byol_augmentation('view1', image_size)
        self.transform2 = get_byol_augmentation('view2', image_size)

    def __call__(self, img):
        return self.transform1(img), self.transform2(img)


print('✓ Augmentation pipeline defined (asymmetric view1/view2)')

### 4.2 Dataset

In [ ]:
# ─── CIFAR-10 Dataset ───
# Data is downloaded to Google Drive so it persists across sessions

# Training set with BYOL pair augmentation (for pretraining)
train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=True,
    transform=BYOLPairTransform(IMAGE_SIZE)
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True
)

# Standard transform for evaluation (no augmentation beyond normalize)
eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                std=[0.2470, 0.2435, 0.2616]),
])

# Separate datasets for linear evaluation
eval_train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=False, transform=eval_transform
)
eval_test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=False, download=True, transform=eval_transform
)

eval_train_loader = DataLoader(
    eval_train_dataset, batch_size=512, shuffle=False,
    num_workers=0, pin_memory=True  # num_workers=0 avoids Colab multiprocessing errors
)
eval_test_loader = DataLoader(
    eval_test_dataset, batch_size=512, shuffle=False,
    num_workers=0, pin_memory=True  # num_workers=0 avoids Colab multiprocessing errors
)

print(f'Training samples: {len(train_dataset)}')
print(f'Test samples: {len(eval_test_dataset)}')
print(f'Batches per epoch: {len(train_loader)}')
print(f'Dataset cached at: {DATA_DIR}')

### 4.3 MLP (Projector & Predictor)

Both the projector and predictor follow the same architecture: `Linear → BatchNorm → ReLU → Linear`.

Key details from the paper and official implementation:
- The **hidden** linear layer has bias and is followed by BatchNorm + ReLU
- The **output** linear layer has no bias (following the official JAX code)
- BatchNorm is applied to the hidden layer only

> ⚠️ **Common Mistake:** Using LayerNorm instead of BatchNorm in the projector/predictor. The paper explicitly uses BatchNorm. While Richemond et al. later showed BYOL can work with other normalizations, the default configuration uses BN. Swapping to LN without adjustment can degrade performance.

In [ ]:
# ─── MLP Module ───

class MLP(nn.Module):
    """Two-layer MLP used for both projector and predictor in BYOL.

    Architecture: Linear(in, hidden) → BN → ReLU → Linear(hidden, out)

    The first linear layer includes bias; the second does not.
    BatchNorm is applied only to the hidden representation.
    This follows the official DeepMind implementation.
    """
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim, bias=True),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim, bias=False),
        )

    def forward(self, x):
        return self.net(x)


print('✓ MLP module defined')

### 4.4 BYOL Model

The BYOL model contains:
1. **Online network:** encoder (ResNet-18) + projector + predictor — trainable via gradient descent
2. **Target network:** encoder (ResNet-18) + projector — updated only via EMA

The target network is initialized as a copy of the online network's encoder + projector (without the predictor).

> ⚙️ **Engineering Decision:** I strip the final `fc` layer from ResNet-18 to use it as a feature extractor. The encoder output is 512-dimensional (ResNet-18's final average pool). The projector maps 512 → 4096 → 256, and the predictor maps 256 → 4096 → 256.

In [ ]:
# ─── BYOL Model ───

class BYOL(nn.Module):
    """Bootstrap Your Own Latent — complete model.

    Online network:  encoder → projector → predictor  (receives gradients)
    Target network:  encoder → projector              (EMA-updated, no gradients)

    The loss is the symmetrized MSE on L2-normalized vectors.
    """
    def __init__(
        self,
        encoder_dim=512,      # ResNet-18 output dimension
        proj_hidden=4096,
        proj_out=256,
        pred_hidden=4096,
        pred_out=256,
    ):
        super().__init__()

        # ── Online network ──
        # Encoder: ResNet-18 without the final classification layer
        resnet = torchvision.models.resnet18(weights=None)
        # For CIFAR-10 (32x32): replace the aggressive 7x7 conv + maxpool
        # with a gentler 3x3 conv. Standard practice for CIFAR-scale SSL.
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        self.online_encoder = nn.Sequential(*list(resnet.children())[:-1], nn.Flatten())

        self.online_projector = MLP(encoder_dim, proj_hidden, proj_out)
        self.online_predictor = MLP(proj_out, pred_hidden, pred_out)

        # ── Target network ──
        # Deep copy of online encoder + projector (no predictor)
        self.target_encoder = copy.deepcopy(self.online_encoder)
        self.target_projector = copy.deepcopy(self.online_projector)

        # Target network does not receive gradients
        for param in self.target_encoder.parameters():
            param.requires_grad = False
        for param in self.target_projector.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def update_target(self, tau):
        """Exponential moving average update of the target network.

        ξ ← τ·ξ + (1-τ)·θ

        Must be called after each optimizer step.
        """
        for online_params, target_params in zip(
            list(self.online_encoder.parameters()) + list(self.online_projector.parameters()),
            list(self.target_encoder.parameters()) + list(self.target_projector.parameters()),
        ):
            target_params.data = tau * target_params.data + (1.0 - tau) * online_params.data

    def forward(self, view1, view2):
        """Forward pass with symmetrized loss.

        Computes loss for both directions:
          - online(view1) predicts target(view2)
          - online(view2) predicts target(view1)

        Returns the mean loss (scalar).
        """
        # ── Online forward (both views) ──
        online_proj_1 = self.online_projector(self.online_encoder(view1))
        online_proj_2 = self.online_projector(self.online_encoder(view2))
        online_pred_1 = self.online_predictor(online_proj_1)
        online_pred_2 = self.online_predictor(online_proj_2)

        # ── Target forward (both views, no gradients) ──
        with torch.no_grad():
            target_proj_1 = self.target_projector(self.target_encoder(view1))
            target_proj_2 = self.target_projector(self.target_encoder(view2))

        # ── Loss: MSE on L2-normalized vectors ──
        loss_12 = self._regression_loss(online_pred_1, target_proj_2)
        loss_21 = self._regression_loss(online_pred_2, target_proj_1)

        return (loss_12 + loss_21).mean()

    @staticmethod
    def _regression_loss(online, target):
        """MSE loss on L2-normalized vectors.

        ||q/||q|| - z'/||z'||| ^2 = 2 - 2 * cos(q, z')

        We use the cosine similarity form for numerical stability.
        """
        online = F.normalize(online, dim=-1)
        target = F.normalize(target, dim=-1)
        return 2.0 - 2.0 * (online * target).sum(dim=-1)


# ── Instantiate and verify ──
model = BYOL(
    encoder_dim=512,
    proj_hidden=PROJ_HIDDEN_DIM,
    proj_out=PROJ_OUTPUT_DIM,
    pred_hidden=PRED_HIDDEN_DIM,
    pred_out=PRED_OUTPUT_DIM,
).to(device)

# Count parameters
online_params = sum(p.numel() for p in model.online_encoder.parameters()) + \
                sum(p.numel() for p in model.online_projector.parameters()) + \
                sum(p.numel() for p in model.online_predictor.parameters())
target_params = sum(p.numel() for p in model.target_encoder.parameters()) + \
                sum(p.numel() for p in model.target_projector.parameters())

print(f'Online network parameters: {online_params:,}')
print(f'Target network parameters: {target_params:,} (not in optimizer)')
print(f'Total model parameters: {online_params + target_params:,}')

### 4.5 EMA Cosine Schedule

The EMA decay $\tau$ follows a cosine schedule from $\tau_{\text{base}}$ to 1.0.

In [ ]:
# ─── EMA Schedule ───

def ema_cosine_schedule(current_step, max_steps, base_tau=0.996):
    """Cosine schedule for EMA decay rate.

    τ_k = 1 - (1 - τ_base) * (cos(πk/K) + 1) / 2

    Starts at τ_base (faster updates), ramps to 1.0 (frozen target).
    """
    return 1.0 - (1.0 - base_tau) * (math.cos(math.pi * current_step / max_steps) + 1.0) / 2.0


# Visualize the schedule
max_steps = NUM_EPOCHS * len(train_loader)
steps = np.arange(max_steps)
taus = [ema_cosine_schedule(s, max_steps, EMA_TAU_BASE) for s in steps]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(steps / len(train_loader), taus, color='#4A90D9', linewidth=2)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('τ (EMA decay)', fontsize=11)
ax.set_title('EMA Cosine Schedule', fontsize=12, fontweight='bold')
ax.axhline(y=EMA_TAU_BASE, color='gray', linestyle='--', alpha=0.5, label=f'τ_base = {EMA_TAU_BASE}')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='τ = 1.0 (frozen)')
ax.legend(fontsize=9)
ax.set_xlim(0, NUM_EPOCHS)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.6 Visualize Augmented Views

Let's visualize the asymmetric augmentation pipeline to build intuition.

In [ ]:
# ─── Visualize Augmented Pairs ───

# Get a batch of raw images for visualization
vis_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR, train=True, download=False, transform=T.ToTensor()
)

# CIFAR-10 class names
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

fig, axes = plt.subplots(3, 8, figsize=(16, 6))

pair_transform = BYOLPairTransform(IMAGE_SIZE)
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)

for i in range(8):
    idx = np.random.randint(len(vis_dataset))
    img, label = vis_dataset[idx]

    # Get a pair from PIL image
    pil_img = T.ToPILImage()(img)
    v1, v2 = pair_transform(pil_img)

    # Denormalize for visualization
    v1_vis = torch.clamp(v1 * std + mean, 0, 1)
    v2_vis = torch.clamp(v2 * std + mean, 0, 1)

    axes[0, i].imshow(img.permute(1, 2, 0).numpy())
    axes[0, i].set_title(cifar10_classes[label], fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(v1_vis.permute(1, 2, 0).numpy())
    axes[1, i].axis('off')

    axes[2, i].imshow(v2_vis.permute(1, 2, 0).numpy())
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=10, rotation=0, labelpad=50)
axes[1, 0].set_ylabel('View 1', fontsize=10, rotation=0, labelpad=50)
axes[2, 0].set_ylabel('View 2', fontsize=10, rotation=0, labelpad=50)

fig.suptitle('BYOL Asymmetric Augmentation Pairs', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

del vis_dataset  # Free memory

<a id='5'></a>
## 5. Training

The training loop is straightforward:
1. Sample a batch of images
2. Forward through BYOL (symmetrized loss)
3. Backprop and update online parameters
4. EMA-update target parameters
5. Step the LR scheduler

We use mixed precision (AMP) for memory efficiency on the T4.

**🛡️ Crash Recovery:** Checkpoints are saved to Google Drive every 5 epochs. If your Colab session disconnects, simply **re-run all cells** — training will automatically resume from the latest checkpoint.

> ⚙️ **Engineering Decision:** I use a cosine LR schedule with linear warmup. The warmup prevents early instability when the randomly-initialized predictor hasn't learned a meaningful mapping yet.

In [ ]:
# ─── Optimizer & Scheduler ───

# Only optimize online network parameters (encoder + projector + predictor)
optimizer = torch.optim.AdamW(
    list(model.online_encoder.parameters()) +
    list(model.online_projector.parameters()) +
    list(model.online_predictor.parameters()),
    lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
)

# Cosine schedule with linear warmup
def lr_lambda(current_step):
    warmup_steps = WARMUP_EPOCHS * len(train_loader)
    total_steps = NUM_EPOCHS * len(train_loader)
    if current_step < warmup_steps:
        return current_step / warmup_steps
    progress = (current_step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Mixed precision — only use on CUDA
if USE_AMP:
    scaler = torch.amp.GradScaler('cuda')
else:
    scaler = None

print(f'Optimizer: AdamW (lr={BASE_LR}, wd={WEIGHT_DECAY})')
print(f'Schedule: Cosine with {WARMUP_EPOCHS}-epoch linear warmup')
print(f'AMP: {"Enabled" if USE_AMP else "Disabled (CPU mode)"}')

In [ ]:
# ─── Checkpoint Resume Logic ───

def find_latest_checkpoint(checkpoint_dir):
    """Find the latest checkpoint in the checkpoint directory."""
    pattern = os.path.join(checkpoint_dir, 'byol_checkpoint_epoch*.pt')
    checkpoints = glob.glob(pattern)
    if not checkpoints:
        return None
    # Extract epoch numbers and find the latest
    def get_epoch(path):
        basename = os.path.basename(path)
        return int(basename.replace('byol_checkpoint_epoch', '').replace('.pt', ''))
    latest = max(checkpoints, key=get_epoch)
    return latest


def save_checkpoint(model, optimizer, scheduler, scaler, epoch, global_step,
                    loss_history, lr_history, checkpoint_dir):
    """Save a complete training checkpoint to Google Drive."""
    checkpoint = {
        'epoch': epoch,
        'global_step': global_step,
        'online_encoder': model.online_encoder.state_dict(),
        'online_projector': model.online_projector.state_dict(),
        'online_predictor': model.online_predictor.state_dict(),
        'target_encoder': model.target_encoder.state_dict(),
        'target_projector': model.target_projector.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict() if scaler is not None else None,
        'loss_history': loss_history,
        'lr_history': lr_history,
    }
    path = os.path.join(checkpoint_dir, f'byol_checkpoint_epoch{epoch}.pt')
    torch.save(checkpoint, path)
    return path


def load_checkpoint(checkpoint_path, model, optimizer, scheduler, scaler, device):
    """Load a checkpoint and restore all training state."""
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model.online_encoder.load_state_dict(checkpoint['online_encoder'])
    model.online_projector.load_state_dict(checkpoint['online_projector'])
    model.online_predictor.load_state_dict(checkpoint['online_predictor'])
    model.target_encoder.load_state_dict(checkpoint['target_encoder'])
    model.target_projector.load_state_dict(checkpoint['target_projector'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    if scaler is not None and checkpoint.get('scaler') is not None:
        scaler.load_state_dict(checkpoint['scaler'])

    return (
        checkpoint['epoch'],
        checkpoint['global_step'],
        checkpoint['loss_history'],
        checkpoint['lr_history'],
    )


# ─── Try to resume from checkpoint ───
start_epoch = 0
global_step = 0
loss_history = []
lr_history = []

latest_ckpt = find_latest_checkpoint(CHECKPOINT_DIR)
if latest_ckpt is not None:
    print(f'🔄 Found checkpoint: {latest_ckpt}')
    start_epoch, global_step, loss_history, lr_history = load_checkpoint(
        latest_ckpt, model, optimizer, scheduler, scaler, device
    )
    print(f'   Resuming from epoch {start_epoch}, global step {global_step}')
    print(f'   Loss history: {len(loss_history)} epochs recorded')
    if start_epoch >= NUM_EPOCHS:
        print(f'✅ Training already complete ({start_epoch}/{NUM_EPOCHS} epochs)!')
else:
    print('🆕 No checkpoint found — starting fresh training')

In [ ]:
# ─── Training Loop (Colab Free-Tier, with checkpoint resume) ───

max_steps = NUM_EPOCHS * len(train_loader)

# Force garbage collection before training
import gc
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()
    print(f'VRAM before training: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB allocated, '
          f'{torch.cuda.memory_reserved(0) / 1e9:.2f} GB reserved')

if start_epoch < NUM_EPOCHS:
    remaining_epochs = NUM_EPOCHS - start_epoch
    print(f'Starting BYOL pretraining: epochs {start_epoch+1}→{NUM_EPOCHS} '
          f'({remaining_epochs} remaining), {len(train_loader)} batches/epoch')
    print(f'Total steps remaining: {remaining_epochs * len(train_loader):,}')
    print('=' * 70)

    training_start = time.time()

    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train()
        epoch_loss = 0.0
        epoch_start = time.time()

        for (view1, view2), _ in train_loader:
            view1 = view1.to(device, non_blocking=True)
            view2 = view2.to(device, non_blocking=True)

            # Forward + loss (with AMP if on CUDA)
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    loss = model(view1, view2)
                # Backward with scaler
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = model(view1, view2)
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            # EMA update of target network
            tau = ema_cosine_schedule(global_step, max_steps, EMA_TAU_BASE)
            model.update_target(tau)

            # Step LR scheduler
            scheduler.step()

            # Logging
            epoch_loss += loss.item()
            lr_history.append(optimizer.param_groups[0]['lr'])
            global_step += 1

        # Epoch summary
        avg_loss = epoch_loss / len(train_loader)
        loss_history.append(avg_loss)
        epoch_time = time.time() - epoch_start
        current_tau = ema_cosine_schedule(global_step - 1, max_steps, EMA_TAU_BASE)

        if (epoch + 1) % 5 == 0 or epoch == start_epoch:
            vram_info = ''
            if device.type == 'cuda':
                vram_info = f' | VRAM: {torch.cuda.memory_allocated(0) / 1e9:.1f}GB'
            print(f'Epoch [{epoch+1:3d}/{NUM_EPOCHS}] | '
                  f'Loss: {avg_loss:.4f} | '
                  f'τ: {current_tau:.6f} | '
                  f'LR: {optimizer.param_groups[0]["lr"]:.6f} | '
                  f'Time: {epoch_time:.1f}s{vram_info}')

        # Checkpoint every CHECKPOINT_EVERY epochs (saved to Drive)
        if (epoch + 1) % CHECKPOINT_EVERY == 0 or (epoch + 1) == NUM_EPOCHS:
            ckpt_path = save_checkpoint(
                model, optimizer, scheduler, scaler,
                epoch + 1, global_step, loss_history, lr_history,
                CHECKPOINT_DIR
            )
            print(f'  💾 Checkpoint saved: {ckpt_path}')

    total_time = time.time() - training_start
    print('=' * 70)
    print(f'Training complete in {total_time/60:.1f} minutes')
    print(f'Final loss: {loss_history[-1]:.4f}')
else:
    print(f'✅ Training already complete — loaded from checkpoint (epoch {start_epoch})')
    total_time = 0

<a id='6'></a>
## 6. Evaluation

### 6.1 Feature Extraction

We extract features from the **online encoder** (before the projector). This follows the standard SSL evaluation protocol: the encoder learns general representations, while the projector/predictor are task-specific heads that are discarded after pretraining.

> 🔬 **Researcher's Note:** The question of *where* to extract representations matters. In SimCLR, Chen et al. showed that representations before the projector ($h$) generalize better than representations after it ($z$). The projector destroys information that's useful for downstream tasks — it learns to be invariant to augmentations, which means it discards information about color, scale, and position. The encoder, by contrast, retains this information because the projector acts as an information bottleneck that absorbs the invariance.

In [ ]:
# ─── Feature Extraction ───

@torch.no_grad()
def extract_features(encoder, data_loader, device):
    """Extract frozen encoder features for all samples.

    Returns:
        features: (N, D) tensor of encoder outputs
        labels: (N,) tensor of ground-truth labels
    """
    encoder.eval()
    all_features = []
    all_labels = []

    for images, labels in tqdm(data_loader, desc='Extracting features', leave=False):
        images = images.to(device, non_blocking=True)
        features = encoder(images)  # (B, 512)
        all_features.append(features.cpu())
        all_labels.append(labels)

    return torch.cat(all_features), torch.cat(all_labels)


print('Extracting features from online encoder...')
train_features, train_labels = extract_features(model.online_encoder, eval_train_loader, device)
test_features, test_labels = extract_features(model.online_encoder, eval_test_loader, device)
print(f'Train features: {train_features.shape}')
print(f'Test features: {test_features.shape}')

### 6.2 Linear Evaluation

The standard evaluation protocol for self-supervised representations: train a single linear layer on top of frozen encoder features. A higher accuracy indicates better representation quality.

We use sklearn's `LogisticRegression` with L-BFGS, which is equivalent to training a linear layer with cross-entropy loss — but converges faster and doesn't require tuning a learning rate.

In [ ]:
# ─── Linear Evaluation ───

print('Training linear classifier on frozen features...')

# Normalize features (standard practice)
train_feats_np = F.normalize(train_features, dim=-1).numpy()
test_feats_np = F.normalize(test_features, dim=-1).numpy()
train_labels_np = train_labels.numpy()
test_labels_np = test_labels.numpy()

# Logistic regression (equivalent to training a linear layer)
linear_clf = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    multi_class='multinomial',
    C=1.0,          # No strong regularization — let the features speak
    verbose=0,
)
linear_clf.fit(train_feats_np, train_labels_np)

train_acc = linear_clf.score(train_feats_np, train_labels_np) * 100
test_acc = linear_clf.score(test_feats_np, test_labels_np) * 100

print(f'\n┌──────────────────────────────────────┐')
print(f'│  LINEAR EVALUATION RESULTS           │')
print(f'├──────────────────────────────────────┤')
print(f'│  Train Accuracy: {train_acc:6.2f}%             │')
print(f'│  Test Accuracy:  {test_acc:6.2f}%             │')
print(f'└──────────────────────────────────────┘')

### 6.3 k-NN Evaluation

k-NN evaluation requires no training at all — it measures how well the embedding space clusters semantically similar images. If an image's nearest neighbors in embedding space are mostly the same class, the representations are good.

> 🔬 **Researcher's Note:** k-NN is a useful *secondary* metric because it's non-parametric. Linear eval can overfit or underfit depending on the regularization and optimizer settings. k-NN has no such issue — it purely measures the geometry of the learned embedding space.

In [ ]:
# ─── k-NN Evaluation ───

print(f'k-NN evaluation (k={KNN_K})...')

knn_clf = KNeighborsClassifier(n_neighbors=KNN_K, metric='cosine', n_jobs=-1)
knn_clf.fit(train_feats_np, train_labels_np)

knn_train_acc = knn_clf.score(train_feats_np, train_labels_np) * 100
knn_test_acc = knn_clf.score(test_feats_np, test_labels_np) * 100

print(f'\n┌──────────────────────────────────────┐')
print(f'│  k-NN EVALUATION RESULTS (k={KNN_K})    │')
print(f'├──────────────────────────────────────┤')
print(f'│  Train Accuracy: {knn_train_acc:6.2f}%             │')
print(f'│  Test Accuracy:  {knn_test_acc:6.2f}%             │')
print(f'└──────────────────────────────────────┘')

### 6.4 Random Baseline Comparison

To contextualize our results, let's compare against a randomly initialized encoder (no pretraining).

In [ ]:
# ─── Random Baseline ───

print('Computing random baseline...')

# Create a fresh, randomly initialized ResNet-18
random_resnet = torchvision.models.resnet18(weights=None)
random_resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
random_resnet.maxpool = nn.Identity()
random_encoder = nn.Sequential(*list(random_resnet.children())[:-1], nn.Flatten()).to(device)

rand_train_feats, rand_train_labels = extract_features(random_encoder, eval_train_loader, device)
rand_test_feats, rand_test_labels = extract_features(random_encoder, eval_test_loader, device)

rand_train_feats_np = F.normalize(rand_train_feats, dim=-1).numpy()
rand_test_feats_np = F.normalize(rand_test_feats, dim=-1).numpy()

rand_clf = LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='multinomial')
rand_clf.fit(rand_train_feats_np, rand_train_labels.numpy())
rand_test_acc = rand_clf.score(rand_test_feats_np, rand_test_labels.numpy()) * 100

print(f'\n┌──────────────────────────────────────────────────┐')
print(f'│  COMPARISON                                      │')
print(f'├──────────────────────────────────────────────────┤')
print(f'│  BYOL Linear Probe:      {test_acc:6.2f}%                │')
print(f'│  BYOL k-NN:              {knn_test_acc:6.2f}%                │')
print(f'│  Random Init Baseline:   {rand_test_acc:6.2f}%                │')
print(f'└──────────────────────────────────────────────────┘')

del random_resnet, random_encoder, rand_train_feats, rand_test_feats  # Free memory
torch.cuda.empty_cache()

<a id='7'></a>
## 7. Visualizations

In [ ]:
# ─── 7.1 Training Loss Curve ───

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Loss curve
ax1.plot(range(1, len(loss_history) + 1), loss_history, color='#E74C3C', linewidth=1.5)
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('BYOL Loss', fontsize=11)
ax1.set_title('Training Loss', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.axvline(x=WARMUP_EPOCHS, color='gray', linestyle='--', alpha=0.5, label=f'Warmup ends (epoch {WARMUP_EPOCHS})')
ax1.legend(fontsize=9)

# Learning rate schedule
ax2.plot(np.arange(len(lr_history)) / len(train_loader), lr_history, color='#3498DB', linewidth=1.5)
ax2.set_xlabel('Epoch', fontsize=11)
ax2.set_ylabel('Learning Rate', fontsize=11)
ax2.set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'training_loss.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'  Saved to {RESULTS_DIR}/training_loss.png')

In [ ]:
# ─── 7.2 t-SNE Visualization ───

print('Computing t-SNE embedding (this takes ~1 minute)...')

# Use a random subset for t-SNE (10K of 50K training samples)
n_tsne = 10000
indices = np.random.choice(len(train_features), n_tsne, replace=False)
tsne_features = F.normalize(train_features[indices], dim=-1).numpy()
tsne_labels = train_labels[indices].numpy()

# CIFAR-10 class names (ensure available in this cell)
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=SEED)
tsne_result = tsne.fit_transform(tsne_features)

# Color palette
colors = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(10, 8))
for i in range(10):
    mask = tsne_labels == i
    ax.scatter(tsne_result[mask, 0], tsne_result[mask, 1],
               c=[colors[i]], label=cifar10_classes[i], s=5, alpha=0.6)

ax.legend(fontsize=9, markerscale=4, loc='best')
ax.set_title('t-SNE of BYOL Representations (Online Encoder)', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE 1', fontsize=11)
ax.set_ylabel('t-SNE 2', fontsize=11)
ax.grid(True, alpha=0.2)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'tsne_embeddings.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'  Saved to {RESULTS_DIR}/tsne_embeddings.png')

In [ ]:
# ─── 7.3 Nearest Neighbor Retrieval ───

print('Computing nearest neighbor retrieval...')

# CIFAR-10 class names
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

# Use test set as queries, train set as gallery
n_queries = 8
n_neighbors = 5

# Compute cosine similarity between test queries and all train features
test_feats_norm = F.normalize(test_features, dim=-1)
train_feats_norm = F.normalize(train_features, dim=-1)

# Pick random queries from different classes
query_indices = []
for cls_id in range(n_queries):
    cls_mask = (test_labels == cls_id).nonzero(as_tuple=True)[0]
    query_indices.append(cls_mask[np.random.randint(len(cls_mask))].item())

fig, axes = plt.subplots(n_queries, n_neighbors + 1, figsize=(12, 16))

# Get raw images for visualization
train_raw = torchvision.datasets.CIFAR10(root=DATA_DIR, train=True, download=False, transform=T.ToTensor())
test_raw = torchvision.datasets.CIFAR10(root=DATA_DIR, train=False, download=False, transform=T.ToTensor())

for row, qi in enumerate(query_indices):
    query_feat = test_feats_norm[qi].unsqueeze(0)  # (1, D)
    similarities = (query_feat @ train_feats_norm.T).squeeze()  # (N_train,)
    _, topk = similarities.topk(n_neighbors)

    # Show query
    q_img, q_label = test_raw[qi]
    axes[row, 0].imshow(q_img.permute(1, 2, 0).numpy())
    axes[row, 0].set_title(f'Query\n{cifar10_classes[q_label]}', fontsize=8, fontweight='bold')
    axes[row, 0].axis('off')
    # Red border for query
    for spine in axes[row, 0].spines.values():
        spine.set_edgecolor('red')
        spine.set_linewidth(2)
        spine.set_visible(True)

    # Show neighbors
    for col, ni in enumerate(topk):
        n_img, n_label = train_raw[ni.item()]
        axes[row, col + 1].imshow(n_img.permute(1, 2, 0).numpy())
        sim_val = similarities[ni].item()
        color = '#2ECC71' if n_label == q_label else '#E74C3C'
        axes[row, col + 1].set_title(f'{cifar10_classes[n_label]}\n({sim_val:.2f})',
                                      fontsize=7, color=color)
        axes[row, col + 1].axis('off')

fig.suptitle('Nearest Neighbor Retrieval (BYOL Embeddings)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'nearest_neighbors.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'  Saved to {RESULTS_DIR}/nearest_neighbors.png')

del train_raw, test_raw  # Free memory

<a id='8'></a>
## 8. Ablation Studies

These ablations directly test the paper's claims about which components are essential.

### 8.1 Ablation: Remove the Predictor (Expect Collapse)

The predictor is the online network's extra capacity. Without it, the online and target networks have identical architectures, and the system has a trivial solution: output a constant vector.

> ❓ **Questions for Future Me:** Does collapse happen immediately, or does the model learn something before collapsing? Is there a phase transition?

In [ ]:
# ─── Ablation 1: No Predictor (Colab-optimized) ───

class BYOL_NoPredictor(nn.Module):
    """BYOL without the predictor — should collapse."""
    def __init__(self, encoder_dim=512, proj_hidden=1024, proj_out=256):
        super().__init__()
        resnet = torchvision.models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        self.online_encoder = nn.Sequential(*list(resnet.children())[:-1], nn.Flatten())
        self.online_projector = MLP(encoder_dim, proj_hidden, proj_out)

        self.target_encoder = copy.deepcopy(self.online_encoder)
        self.target_projector = copy.deepcopy(self.online_projector)
        for p in self.target_encoder.parameters(): p.requires_grad = False
        for p in self.target_projector.parameters(): p.requires_grad = False

    @torch.no_grad()
    def update_target(self, tau):
        for op, tp in zip(
            list(self.online_encoder.parameters()) + list(self.online_projector.parameters()),
            list(self.target_encoder.parameters()) + list(self.target_projector.parameters()),
        ):
            tp.data = tau * tp.data + (1.0 - tau) * op.data

    def forward(self, view1, view2):
        online_z1 = self.online_projector(self.online_encoder(view1))
        online_z2 = self.online_projector(self.online_encoder(view2))
        with torch.no_grad():
            target_z1 = self.target_projector(self.target_encoder(view1))
            target_z2 = self.target_projector(self.target_encoder(view2))

        loss_12 = 2.0 - 2.0 * (F.normalize(online_z1, dim=-1) * F.normalize(target_z2, dim=-1)).sum(dim=-1)
        loss_21 = 2.0 - 2.0 * (F.normalize(online_z2, dim=-1) * F.normalize(target_z1, dim=-1)).sum(dim=-1)
        return (loss_12 + loss_21).mean()


# Short training run to observe collapse
ABLATION_EPOCHS = 15
print(f'Ablation 1: BYOL without predictor ({ABLATION_EPOCHS} epochs)')
print('Expected: representational collapse')
print('-' * 50)

abl1_model = BYOL_NoPredictor().to(device)
abl1_optimizer = torch.optim.AdamW(
    list(abl1_model.online_encoder.parameters()) +
    list(abl1_model.online_projector.parameters()),
    lr=BASE_LR, weight_decay=WEIGHT_DECAY
)
if USE_AMP:
    abl1_scaler = torch.amp.GradScaler('cuda')
else:
    abl1_scaler = None
abl1_losses = []
abl1_max_steps = ABLATION_EPOCHS * len(train_loader)

for epoch in range(ABLATION_EPOCHS):
    abl1_model.train()
    epoch_loss = 0.0
    step = epoch * len(train_loader)
    for (v1, v2), _ in train_loader:
        v1, v2 = v1.to(device), v2.to(device)
        if USE_AMP:
            with torch.amp.autocast('cuda'):
                loss = abl1_model(v1, v2)
            abl1_optimizer.zero_grad(set_to_none=True)
            abl1_scaler.scale(loss).backward()
            abl1_scaler.step(abl1_optimizer)
            abl1_scaler.update()
        else:
            loss = abl1_model(v1, v2)
            abl1_optimizer.zero_grad(set_to_none=True)
            loss.backward()
            abl1_optimizer.step()
        tau = ema_cosine_schedule(step, abl1_max_steps, EMA_TAU_BASE)
        abl1_model.update_target(tau)
        epoch_loss += loss.item()
        step += 1
    avg_loss = epoch_loss / len(train_loader)
    abl1_losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch [{epoch+1}/{ABLATION_EPOCHS}] Loss: {avg_loss:.4f}')

# Evaluate
abl1_train_feats, abl1_train_labels = extract_features(abl1_model.online_encoder, eval_train_loader, device)
abl1_test_feats, abl1_test_labels = extract_features(abl1_model.online_encoder, eval_test_loader, device)

abl1_clf = LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='multinomial')
abl1_clf.fit(F.normalize(abl1_train_feats, dim=-1).numpy(), abl1_train_labels.numpy())
abl1_acc = abl1_clf.score(F.normalize(abl1_test_feats, dim=-1).numpy(), abl1_test_labels.numpy()) * 100
print(f'\n  No-predictor linear probe accuracy: {abl1_acc:.2f}%')
print(f'  Full BYOL linear probe accuracy:    {test_acc:.2f}%')

feat_std = abl1_test_feats.std(dim=0).mean().item()
print(f'  Feature std (per-dim mean): {feat_std:.6f}  (near-zero indicates collapse)')

del abl1_model, abl1_optimizer
torch.cuda.empty_cache()

### 8.2 Ablation: EMA τ Sensitivity

How sensitive is BYOL to the EMA decay rate? We test $\tau \in \{0.99, 0.996, 0.999\}$ with short training runs.

- **τ = 0.99:** Target updates relatively fast → may cause instability
- **τ = 0.996:** Paper default → smooth, stable training
- **τ = 0.999:** Target barely moves → too stale, slow convergence

In [ ]:
# ─── Ablation 2: EMA τ Sensitivity (Colab-optimized) ───

TAU_VALUES = [0.99, 0.996, 0.999]  # Dropped 0.9 (obviously too fast)
TAU_ABLATION_EPOCHS = 15
tau_results = {}

print(f'Ablation 2: EMA τ sensitivity ({TAU_ABLATION_EPOCHS} epochs each)')
print('=' * 50)

for tau_base in TAU_VALUES:
    print(f'\n  τ_base = {tau_base}')

    abl_model = BYOL(
        encoder_dim=512, proj_hidden=PROJ_HIDDEN_DIM, proj_out=PROJ_OUTPUT_DIM,
        pred_hidden=PRED_HIDDEN_DIM, pred_out=PRED_OUTPUT_DIM
    ).to(device)

    abl_opt = torch.optim.AdamW(
        list(abl_model.online_encoder.parameters()) +
        list(abl_model.online_projector.parameters()) +
        list(abl_model.online_predictor.parameters()),
        lr=BASE_LR, weight_decay=WEIGHT_DECAY
    )
    if USE_AMP:
        abl_scaler = torch.amp.GradScaler('cuda')
    else:
        abl_scaler = None
    abl_max_steps = TAU_ABLATION_EPOCHS * len(train_loader)
    abl_losses = []

    for epoch in range(TAU_ABLATION_EPOCHS):
        abl_model.train()
        ep_loss = 0.0
        step = epoch * len(train_loader)
        for (v1, v2), _ in train_loader:
            v1, v2 = v1.to(device), v2.to(device)
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    loss = abl_model(v1, v2)
                abl_opt.zero_grad(set_to_none=True)
                abl_scaler.scale(loss).backward()
                abl_scaler.step(abl_opt)
                abl_scaler.update()
            else:
                loss = abl_model(v1, v2)
                abl_opt.zero_grad(set_to_none=True)
                loss.backward()
                abl_opt.step()
            tau = ema_cosine_schedule(step, abl_max_steps, tau_base)
            abl_model.update_target(tau)
            ep_loss += loss.item()
            step += 1
        abl_losses.append(ep_loss / len(train_loader))

    # Evaluate
    abl_feats, abl_labs = extract_features(abl_model.online_encoder, eval_test_loader, device)
    abl_train_feats, abl_train_labs = extract_features(abl_model.online_encoder, eval_train_loader, device)

    clf = LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='multinomial')
    clf.fit(F.normalize(abl_train_feats, dim=-1).numpy(), abl_train_labs.numpy())
    acc = clf.score(F.normalize(abl_feats, dim=-1).numpy(), abl_labs.numpy()) * 100

    tau_results[tau_base] = {'accuracy': acc, 'losses': abl_losses}
    print(f'  → Linear probe accuracy: {acc:.2f}%')

    del abl_model, abl_opt
    torch.cuda.empty_cache()

print('\n' + '=' * 50)
print('Summary:')
for tau_base, result in tau_results.items():
    print(f'  τ_base={tau_base:.3f}  →  {result["accuracy"]:.2f}%')

In [ ]:
# ─── Ablation Plots ───

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 8.1: No-predictor vs full BYOL loss curves
ax1.plot(range(1, len(abl1_losses) + 1), abl1_losses, 'r-', linewidth=2, label='No Predictor (collapsed)')
ax1.plot(range(1, min(len(loss_history), len(abl1_losses)) + 1),
         loss_history[:len(abl1_losses)], 'b-', linewidth=2, label='Full BYOL')
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('BYOL Loss', fontsize=11)
ax1.set_title('Ablation: Predictor Removal', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# 8.2: EMA sensitivity bar chart
tau_labels = [f'τ={t}' for t in TAU_VALUES]
tau_accs = [tau_results[t]['accuracy'] for t in TAU_VALUES]
bar_colors = ['#E74C3C' if a < 40 else '#F39C12' if a < 70 else '#2ECC71' for a in tau_accs]
ax2.bar(tau_labels, tau_accs, color=bar_colors, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Linear Probe Accuracy (%)', fontsize=11)
ax2.set_title('Ablation: EMA τ Sensitivity', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 100)
for i, (label, acc) in enumerate(zip(tau_labels, tau_accs)):
    ax2.text(i, acc + 2, f'{acc:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'ablation_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'  Saved to {RESULTS_DIR}/ablation_results.png')

<a id='9'></a>
## 9. Critical Review

### Strengths
1. **Removes batch size dependence.** BYOL works at batch size 256 without performance degradation — SimCLR loses ~10% accuracy going from 4096 to 256.
2. **Augmentation robustness.** The paper shows that removing individual augmentations hurts BYOL less than SimCLR, suggesting that BYOL representations are less tied to the specific augmentation pipeline.
3. **Elegant simplicity.** The method is conceptually clean: predict one view from another, with an EMA target. No memory banks, no clustering, no specialized loss functions.
4. **Strong transfer performance.** BYOL outperforms on 11/15 transfer benchmarks, suggesting the representations are genuinely general rather than tuned to ImageNet.

### Weaknesses
1. **Collapse prevention is poorly understood.** The paper demonstrates empirically that the predictor + EMA + stop-gradient combination prevents collapse, but doesn't provide a theoretical analysis of *why*. This was partially addressed by follow-up work (Tian et al., 2021; Richemond et al., 2020).
2. **Hyperparameter sensitivity.** The EMA schedule, predictor architecture, and augmentation pipeline must all be carefully tuned. The τ ablation shows that values outside a narrow range degrade performance significantly.
3. **2× compute per step.** Both online and target networks process both views, so each forward pass is ~2× the cost of a contrastive method (which processes each view once).

### The BatchNorm Controversy

Shortly after BYOL's publication, Richemond et al. (2020) raised a pointed question: *does BatchNorm implicitly introduce a contrastive signal?* BatchNorm computes mean and variance across the batch, which means each sample's normalized representation depends on all other samples in the batch. This is a form of information leakage that could function as an implicit negative mechanism.

The subsequent paper "BYOL Works Even Without Batch Statistics" (Richemond et al., 2020) showed that replacing BatchNorm with GroupNorm or LayerNorm still works — but required careful retuning. This suggests that BatchNorm *helps* but is not the sole explanation for why BYOL avoids collapse.

### Historical Impact

BYOL opened the floodgates for **non-contrastive** self-supervised learning:

| Paper | Year | Relationship |
|---|---|---|
| SimSiam | 2021 | Removed EMA — showed that stop-gradient + predictor alone suffices |
| Barlow Twins | 2021 | Replaced prediction with cross-correlation → decorrelation loss |
| VICReg | 2022 | Explicitly decomposed non-collapse into variance, invariance, covariance |
| DINO | 2021 | Extended teacher-student framework to ViTs with self-distillation + centering |

Each of these methods owes a direct intellectual debt to BYOL's central insight: *the repulsive force in contrastive learning is a sufficient but not necessary condition for learning non-trivial representations.*

> 📓 **Research Diary:** Reading the BYOL → SimSiam progression is one of the most elegant research arcs I've encountered. BYOL proves negatives are unnecessary. SimSiam proves *momentum* is unnecessary. If you strip away everything — negatives, momentum, memory banks — what's the minimal sufficient mechanism? It turns out to be: a stop-gradient and a predictor. Two lines of code prevent collapse. The simplicity of this answer is almost unsettling.

<a id='10'></a>
## 10. Reflection & Future Work

### What I Learned

BYOL changed how I think about loss functions in self-supervised learning. Before this paper, I assumed that the contrastive loss — pulling positives together, pushing negatives apart — was the irreducible core of SSL. BYOL shows that only the "pulling together" part is doing the real work. The "pushing apart" was a regularizer that happened to also constrain the method to large batch sizes.

The deeper lesson is epistemological: *question every component of a method, especially the ones that seem load-bearing.* The hardest parts to remove are often the least necessary.

### Future Experiments

1. **SimSiam:** Remove the EMA entirely. Does stop-gradient + predictor alone suffice on CIFAR-10?
2. **Augmentation ablation:** Test BYOL with fewer augmentations — does it degrade more gracefully than SimCLR?
3. **Feature probing at different layers:** Do earlier ResNet layers encode different information than later ones?
4. **Transfer to CIFAR-100:** Does BYOL's pretraining on CIFAR-10 transfer to a harder classification task?
5. **Collapse dynamics:** Train BYOL without the predictor for 200 epochs and monitor the representation covariance matrix — when exactly does collapse occur?

> ❓ **Questions for Future Me:**
> - Is there a theoretical framework that unifies BYOL, SimSiam, VICReg, and Barlow Twins?
> - Can non-contrastive methods scale to CLIP-level multimodal pretraining?
> - What is the minimum EMA schedule length needed for stable training?
> - Does the predictor learn a meaningful transformation, or does it just prevent collapse through capacity asymmetry?

In [ ]:
# ─── Final Summary ───

print('\n' + '═' * 60)
print('  BYOL REPRODUCTION — FINAL RESULTS')
print('═' * 60)
print(f'\n  Encoder:          ResNet-18')
print(f'  Dataset:          CIFAR-10')
print(f'  Pretraining:      {NUM_EPOCHS} epochs, BS={BATCH_SIZE}')
print(f'  Final loss:       {loss_history[-1]:.4f}')
print(f'\n  Linear Probe:     {test_acc:.2f}%')
print(f'  k-NN (k={KNN_K}):      {knn_test_acc:.2f}%')
print(f'  Random Baseline:  {rand_test_acc:.2f}%')
print(f'\n  Ablation Results:')
print(f'    No predictor:   {abl1_acc:.2f}% (collapse expected)')
for tau_base, result in tau_results.items():
    marker = ' ◄ paper default' if tau_base == 0.996 else ''
    print(f'    τ={tau_base:.3f}:       {result["accuracy"]:.2f}%{marker}')
print(f'\n  All results saved to: {RESULTS_DIR}')
print('═' * 60)

# Save final results as JSON for easy access later
final_results = {
    'epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'final_loss': loss_history[-1],
    'linear_probe_test': test_acc,
    'knn_test': knn_test_acc,
    'random_baseline': rand_test_acc,
    'ablation_no_predictor': abl1_acc,
    'ablation_tau': {str(k): v['accuracy'] for k, v in tau_results.items()},
}
with open(os.path.join(RESULTS_DIR, 'final_results.json'), 'w') as f:
    json.dump(final_results, f, indent=2)
print(f'\nResults saved to {RESULTS_DIR}/final_results.json')